In [1]:
# Imports

# For basic use, you only need to import the ETC object
from uvex_imager_etc.etc import ETC

# ETC takes synphot source spectra as inputs
from synphot import SourceSpectrum
from synphot.models import ConstFlux1D, BlackBodyNorm1D

# Coordinates and times are handled using Astropy SkyCoord and Time objects
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time

First we'll define some inputs. 

Input sources can be given either in magnitudes (in which case a flat spectrum is assumed) or as a synphot SourceSpectrum object. A list of multiple SourceSpectrum objects or multiple magnitudes is supported.

Coordinates and observation times can be defined as one for each input source, one common value for all input sources, or alternatively as multiple coordinates and/or observation times for a single input source. Where any of these inputs are multiple, they must have the same length.

If a coordinate or observation time aren't defined, they are set to an arbitrary location and time giving a fairly typical sky background.

In [2]:
# Define some input sources 
source1 = SourceSpectrum(BlackBodyNorm1D, temperature=8000*u.K) 
source2 = SourceSpectrum(ConstFlux1D, amplitude=24*u.ABmag)
source3 = [source1, source2]
source4 = [23.,24.,25.,26.] * u.ABmag

# Define associated coordinates
coord1 = SkyCoord([120.], [15.], unit=u.deg, frame='galactic')
coord2 = SkyCoord([100.,120.,140.], [30.,15.,-40.], unit=u.deg, frame='galactic')
coord3 = SkyCoord(['05:23:34.6 -69:45:22', '00:52:38.0 -72:48:01'], unit=(u.hourangle, u.deg), frame='icrs')
coord4 = SkyCoord([13.4979717], [47.1952583], unit=u.deg, frame='icrs')

# Definte associated observation times
obstime1 = Time(['2030-02-01 09:00:00'], scale='utc', format='iso')
obstime2 = Time(['2030-06-01 09:00:00'], scale='utc', format='iso')
obstime3 = Time(['2030-07-28 09:00:00', '2030-08-05 12:00:00'], scale='utc', format='iso')
obstime4 = Time(['2030-02-01','2030-03-01','2030-04-01','2030-05-01'], scale='utc', format='iso')

To run the ETC, initialize an ETC object with the input source, coordinates, and observation times.

The ETC object has a number of functions that can then be called to perform calculations related to the source, coordinate and time inputs:
- get_snr(exptime, n_frames, band) or get_snr(n_dwells, band)
- get_exposure(snr, band)
- get_dwells(snr, band)
- get_source_count_rate(band) and get_background_count_rate(band)
- get_limiting_mag(snr, exptime, n_frames, band) or get_limiting_mag(snr, n_dwells, band) - this function does not require an input source

In [3]:
etc1 = ETC(source=source1, coordinate=coord1, obstime=obstime1)

# get_info() returns some basic information about the current setup
etc1.get_info()
print('---')

# get_snr() can take either a number of frames at a given exposure time
# or a number of standard observing dwells set to UVEX's default survey observing mode
snr_exp_nuv = etc1.get_snr(exptime=300*u.s, n_frames=2, band='NUV')
snr_dwell_nuv = etc1.get_snr(n_dwells=1, band='NUV')
snr_dwell_fuv = etc1.get_snr(n_dwells=2, band='FUV')

print('NUV SNR in 2x300s: ', snr_exp_nuv)
print('NUV SNR in one dwell: ', snr_dwell_nuv)
print('FUV SNR in two dwells: ', snr_dwell_fuv)

/Users/hannah/Dropbox/PhD/Python/uvex-imager-etc/uvex_imager_etc/uvex.py:43: UserWarning: Multiple CALDBs available for 2026-08-13 00:00:00.000; using 20260813_v0.1c
  warnings.warn(f"Multiple CALDBs available for {latest}; using {self.caldb}")


UVEX version: 20260813_v0.1c
Source: User-defined spectrum
Source position: <SkyCoord (Galactic): (l, b) in deg
    [(120., 15.)]>
Observation time: ['2030-02-01 09:00:00.000']
---
NUV SNR in 2x300s:  [516.23435928]
NUV SNR in one dwell:  [632.25538397]
FUV SNR in two dwells:  [172.02388852]


In [4]:
etc2 = ETC(source=source2, coordinate=coord2, obstime=obstime2)
etc2.get_info()
print('---')

# get_exposure() returns the required exposure time for a single observation to reach the desired SNR
exptime_fuv = etc2.get_exposure(snr=5, band='FUV')

# get_dwells() returns the number of standard UVEX observing dwells to reach the desired SNR 
# i.e. how many telescope visits would be required
n_dwells_nuv = etc2.get_dwells(snr=20, band='NUV')

print('Exposure time to reach SNR=5 in FUV: ', exptime_fuv)
print('Number of dwells to reach SNR=20 in NUV: ', n_dwells_nuv)

/Users/hannah/Dropbox/PhD/Python/uvex-imager-etc/uvex_imager_etc/uvex.py:43: UserWarning: Multiple CALDBs available for 2026-08-13 00:00:00.000; using 20260813_v0.1c
  warnings.warn(f"Multiple CALDBs available for {latest}; using {self.caldb}")


UVEX version: 20260813_v0.1c
Source: User-defined spectrum
Source position: <SkyCoord (Galactic): (l, b) in deg
    [(100.,  30.), (120.,  15.), (140., -40.)]>
Observation time: ['2030-06-01 09:00:00.000']
---
Exposure time to reach SNR=5 in FUV:  [607.25904986 622.51520749 619.02223287] s
Number of dwells to reach SNR=20 in NUV:  [ 7.  8. 12.]


In [5]:
etc3 = ETC(source=source3, coordinate=coord3, obstime=obstime3)
etc3.get_info()
print('---')

# Source and background count rates can also be directly queried
nuv_rate = etc3.get_source_count_rate(band='nuv')
fuv_bg_rate = etc3.get_background_count_rate(band='fuv')

print('NUV source count rate: ', nuv_rate)
print('FUV background count rate: ', fuv_bg_rate)

UVEX version: 20260813_v0.1c
Source: User-defined spectra x 2
Source position: <SkyCoord (ICRS): (ra, dec) in deg
    [(80.89416667, -69.75611111), (13.15833333, -72.80027778)]>
Observation time: ['2030-07-28 09:00:00.000' '2030-08-05 12:00:00.000']
---
NUV source count rate:  [4.44413097e+02 1.65804586e-01] electron / s
FUV background count rate:  [0.0034692  0.00305997] electron / s


In [12]:
etc4 = ETC(source=None, coordinate=coord4, obstime=obstime4)
etc4.get_info()
print('---')

# get_limiting_mag() can be used without an input source
lim_mag_nuv = etc4.get_limiting_mag(snr=10,exptime=900*u.s,n_frames=2,band='nuv')
lim_mag_fuv = etc4.get_limiting_mag(snr=5,n_dwells=2,band='fuv')

print('NUV limiting magnitude for SNR=10, 2x900s: ', lim_mag_nuv)
print('FUV limiting magnitude for SNR=5, 2 dwells', lim_mag_fuv)
print('---')

# New sources and coordinates or observation times can be loaded in using
# set_source(), set_coord(), and set_obstime()
etc4.set_source(source4)
etc4.get_info()
print('---')

snr_dwell_nuv4 = etc4.get_snr(n_dwells=2, band='NUV')

print('NUV SNR in two dwells:', snr_dwell_nuv4)
print('---')

UVEX version: 20260813_v0.1c
Source: None
Source position: <SkyCoord (ICRS): (ra, dec) in deg
    [(13.4979717, 47.1952583)]>
Observation time: ['2030-02-01 00:00:00.000' '2030-03-01 00:00:00.000'
 '2030-04-01 00:00:00.000' '2030-05-01 00:00:00.000']
---
NUV limiting magnitude for SNR=10, 2x900s:  [24.19198752 24.15157858 24.05342106 23.99229812] mag(AB)
FUV limiting magnitude for SNR=5, 2 dwells [24.71845458 24.71741132 24.71449643 24.7123738 ] mag(AB)
---
UVEX version: 20260813_v0.1c
Source: Constant spectrum at [23. 24. 25. 26.] mag(AB) x 4
Source position: <SkyCoord (ICRS): (ra, dec) in deg
    [(13.4979717, 47.1952583)]>
Observation time: ['2030-02-01 00:00:00.000' '2030-03-01 00:00:00.000'
 '2030-04-01 00:00:00.000' '2030-05-01 00:00:00.000']
---
NUV SNR in two dwells: [20.89374055 10.0893821   4.18939427  1.65935397]
---


The ETC will default to using the most recent and latest versioned CALDB in the uvex_imager_etc/response_files directory. This CALDB is generated via uvex_response and can be obtained from the UVEX website - make sure you are using the most up-to-date version for accurate results.

Should you have multiple versions of the CALDB present in response_files and wish to compare with an older version or replicate specific results, you can load an older telescope configuration by initializing a UVEX object and specifying the CALDB folder name (which always takes the form of a date followed by a version). This UVEX object can then be passed into the ETC via the telescope parameter.

In [13]:
from uvex_imager_etc.uvex import UVEX
uvex_old = UVEX(caldb='20260813_v0.1a')
etc5 = ETC(source=source4, coordinate=coord4, obstime=obstime4, telescope=uvex_old)
etc5.get_info()

snr_dwell_nuv5 = etc5.get_snr(n_dwells=2, band='NUV')

print('NUV SNR in two dwells:', snr_dwell_nuv5)

NUV SNR in two dwells: [22.37422863 10.98393844  4.67044541  1.8770972 ]
